# Puma Women's Footwear Scraper

Objective:
Scrape product data from Puma India's Women's Shoes category.

Data Extracted:
- Product URL
- Product Name
- Brand
- Sale Price
- MRP


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print("All imports successful")


All imports successful


In [5]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}


In [6]:
url = "https://in.puma.com/in/en/womens/womens-shoes"

response = requests.get(url, headers=HEADERS, timeout=10)
print("Status Code:", response.status_code)


Status Code: 200


In [8]:
soup = BeautifulSoup(response.text, "html.parser")


In [15]:
products = soup.select("[data-test-id='product-list-item']")
print("Number of products found:", len(products))


Number of products found: 24


In [14]:
all_test_ids = soup.select("[data-test-id]")
len(all_test_ids)


697

In [16]:
extracted_data = []

for product in products:
    try:
        # Find product link
        a_tag = product.find("a", href=True)
        link = a_tag["href"] if a_tag else None
        full_url = "https://in.puma.com" + link if link else None

        # Product name (accessible label)
        product_name = a_tag.get("aria-label") if a_tag else None

        # Price extraction
        sale_price = None
        mrp = None

        for span in product.find_all("span"):
            text = span.get_text(strip=True)
            if "₹" in text and sale_price is None:
                sale_price = text
            elif "₹" in text:
                mrp = text

        extracted_data.append({
            "URL": full_url,
            "Product Name": product_name,
            "Brand": "Puma",
            "Sale Price": sale_price,
            "MRP": mrp
        })

    except Exception:
        continue

print("Extracted products:", len(extracted_data))


Extracted products: 24


In [17]:
all_products = []
page = 1

while True:
    page_url = f"https://in.puma.com/in/en/womens/womens-shoes?page={page}"
    response = requests.get(page_url, headers=HEADERS, timeout=10)

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")
    products = soup.select("[data-test-id='product-list-item']")

    if not products:
        break

    for product in products:
        try:
            a_tag = product.find("a", href=True)
            link = a_tag["href"] if a_tag else None
            full_url = "https://in.puma.com" + link if link else None

            product_name = a_tag.get("aria-label") if a_tag else None

            sale_price = None
            mrp = None
            for span in product.find_all("span"):
                text = span.get_text(strip=True)
                if "₹" in text and sale_price is None:
                    sale_price = text
                elif "₹" in text:
                    mrp = text

            all_products.append({
                "URL": full_url,
                "Product Name": product_name,
                "Brand": "Puma",
                "Sale Price": sale_price,
                "MRP": mrp
            })

        except Exception:
            continue

    print(f"Page {page}: {len(products)} products scraped")
    page += 1
    time.sleep(1)

print("Total products collected:", len(all_products))


Page 1: 24 products scraped
Page 2: 24 products scraped
Page 3: 24 products scraped
Page 4: 24 products scraped
Page 5: 24 products scraped
Page 6: 24 products scraped
Page 7: 24 products scraped
Page 8: 24 products scraped
Page 9: 24 products scraped
Page 10: 24 products scraped
Page 11: 24 products scraped
Page 12: 24 products scraped
Page 13: 24 products scraped
Page 14: 24 products scraped
Page 15: 24 products scraped
Page 16: 24 products scraped
Page 17: 24 products scraped
Page 18: 24 products scraped
Page 19: 24 products scraped
Page 20: 24 products scraped
Page 21: 24 products scraped
Page 22: 24 products scraped
Page 23: 24 products scraped
Page 24: 24 products scraped
Page 25: 24 products scraped
Page 26: 24 products scraped
Page 27: 24 products scraped
Page 28: 24 products scraped
Page 29: 22 products scraped
Total products collected: 694


In [18]:
df = pd.DataFrame(all_products)

# Remove duplicates using URL as unique key
df.drop_duplicates(subset="URL", inplace=True)

df.reset_index(drop=True, inplace=True)

df.head()


,URL,Product Name,Brand,Sale Price,MRP
0,https://in.puma.com/in/en/pd/skyrocket-lite-wo...,"Skyrocket Lite Women's Running Shoes, -30%",Puma,₹0,None
1,https://in.puma.com/in/en/pd/galaxis-pro-women...,"3 Colors, Galaxis Pro Women's Performance Boos...",Puma,₹0,None
2,https://in.puma.com/in/en/pd/mayze-leather-wom...,"2 Colors, Mayze Leather Women's Sneakers, -52%",Puma,₹0,None
3,https://in.puma.com/in/en/pd/galaxis-pro-women...,"3 Colors, Galaxis Pro Women's Performance Boos...",Puma,₹0,None
4,https://in.puma.com/in/en/pd/carina-slim-perf-...,"2 Colors, Carina Slim Perf Women's Sneakers, -20%",Puma,₹0,None


In [19]:
df.to_csv("dataset.csv", index=False)
print("dataset.csv saved successfully")


dataset.csv saved successfully
